# Causal Early Warning Audit

Ce notebook reprend le script `causal_early_warning_audit.py`.

But : rendre l'experience lisible et executable dans Jupyter, avec des explications simples en francais.

Utilisation : executez les cellules dans l'ordre. La derniere cellule lance le script avec des arguments controles.


## Pertinence pour le live streaming

Decision : garde. Audit explicitement live-stream: decision a t avec seulement l'historique jusqu'a t.

Regle appliquee : l'experience doit aider a entrainer, choisir, calibrer, tester ou executer une prediction en flux video avec seulement les informations disponibles a l'instant courant.


## Pourquoi ce notebook est garde

- Description du script : Causal pre-entry early-warning audit for final risk scores.
- Run par defaut : `runs/exp_092_causal_early_warning_audit`.

Decision : garde, car il correspond a un artefact experimental, une commande de reproduction, ou un audit du catalogue.


## Avant de commencer

- Verifiez que les donnees et les dossiers `runs/` attendus existent.
- Le notebook n'a pas ete execute pendant sa creation.
- Les cellules de lancement creent un nom ou un dossier unique quand cela evite d'ecraser un resultat existant.


In [ ]:
# Compatibilite Jupyter
# Certains scripts utilisent __file__. Dans un notebook, on le definit explicitement.
from pathlib import Path

PROJECT_ROOT = Path.cwd()
__file__ = str(PROJECT_ROOT / "causal_early_warning_audit.py")


## Importations et configuration

Cette cellule charge les bibliotheques et definit les constantes utilisees par le script.

In [ ]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd

from ml_pipeline import ROOT, alarm_episodes, write_json
from sequence_experiments import make_run_dir


POLICIES = {
    "max_1s_early_low_fa": ["early_1s_recall", "false_alarms_per_min", "pre_entry_recall", "event_precision"],
    "balanced_event_f1": ["event_f1", "early_1s_recall", "false_alarms_per_min", "event_precision"],
    "low_false_alarm": ["false_alarms_per_min", "event_precision", "pre_entry_recall", "early_1s_recall"],
}


## Fonction `resolve`

Cette cellule definit `resolve`. Elle prepare une partie du script.

In [ ]:
def resolve(path):
    path = Path(path)
    if not path.is_absolute():
        path = ROOT / path
    return path


## Fonction `load_final_scores`

Cette cellule definit `load_final_scores`. Elle prepare une partie du script.

In [ ]:
def load_final_scores(final_score_run):
    frames = []
    for path in sorted((final_score_run / "features").glob("final_scores_seed*.csv")):
        seed = int(path.stem.replace("final_scores_seed", ""))
        df = pd.read_csv(path)
        df["repeat_seed"] = seed
        frames.append(df)
    if not frames:
        raise SystemExit(f"No final score tables found in {final_score_run / 'features'}")
    return pd.concat(frames, ignore_index=True)


## Fonction `safe_float`

Cette cellule definit `safe_float`. Elle prepare une partie du script.

In [ ]:
def safe_float(value):
    if value is None:
        return np.nan
    try:
        out = float(value)
    except (TypeError, ValueError):
        return np.nan
    return out


## Fonction `evaluate_split`

Cette cellule definit `evaluate_split`. Elle prepare une partie du script.

In [ ]:
def evaluate_split(df, score_col, threshold, split_name, seed, persistence_windows):
    split_df = df[(df["split"] == split_name) & (df["repeat_seed"] == seed)].copy()
    if split_df.empty:
        return None

    tp_pre = 0
    tp_early_1s = 0
    missed = 0
    early_times = []
    false_alarm_episodes = 0
    negative_minutes = 0.0
    danger_videos = 0
    negative_videos = 0

    for video_id, group in split_df.groupby("video_id", sort=False):
        group = group.sort_values("time_s").copy()
        times = group["time_s"].to_numpy(dtype=float)
        scores = group[score_col].to_numpy(dtype=float)
        alarms = alarm_episodes(times, scores, threshold, gap_s=1.0, persistence_windows=persistence_windows)
        is_danger = int(group["is_danger_clip"].max()) == 1
        target = safe_float(group["target_time_s"].dropna().iloc[0]) if group["target_time_s"].notna().any() else np.nan

        if is_danger and not np.isnan(target):
            danger_videos += 1
            pre_entry = [float(t) for t in alarms if float(t) < target]
            if pre_entry:
                first = min(pre_entry)
                early_s = float(target - first)
                early_times.append(early_s)
                tp_pre += 1
                if early_s >= 1.0:
                    tp_early_1s += 1
            else:
                missed += 1
        else:
            negative_videos += 1
            false_alarm_episodes += len(alarms)
            if len(times):
                negative_minutes += max(0.0, float(times.max() - times.min())) / 60.0

    pre_entry_recall = tp_pre / danger_videos if danger_videos else np.nan
    early_1s_recall = tp_early_1s / danger_videos if danger_videos else np.nan
    false_alarms_per_min = false_alarm_episodes / negative_minutes if negative_minutes > 0 else 0.0
    event_precision = tp_pre / (tp_pre + false_alarm_episodes) if (tp_pre + false_alarm_episodes) else np.nan
    event_f1 = (
        2 * event_precision * pre_entry_recall / (event_precision + pre_entry_recall)
        if event_precision == event_precision and pre_entry_recall == pre_entry_recall and (event_precision + pre_entry_recall) > 0
        else np.nan
    )
    return {
        "repeat_seed": seed,
        "split": split_name,
        "score_variant": score_col,
        "threshold": float(threshold),
        "danger_videos": danger_videos,
        "negative_videos": negative_videos,
        "pre_entry_detected": tp_pre,
        "early_1s_detected": tp_early_1s,
        "missed_entries": missed,
        "false_alarm_episodes": false_alarm_episodes,
        "negative_minutes": negative_minutes,
        "pre_entry_recall": pre_entry_recall,
        "early_1s_recall": early_1s_recall,
        "event_precision": event_precision,
        "event_f1": event_f1,
        "false_alarms_per_min": false_alarms_per_min,
        "median_early_warning_s": float(np.median(early_times)) if early_times else np.nan,
        "mean_early_warning_s": float(np.mean(early_times)) if early_times else np.nan,
        "min_early_warning_s": float(np.min(early_times)) if early_times else np.nan,
    }


## Fonction `summarize`

Cette cellule definit `summarize`. Elle prepare une partie du script.

In [ ]:
def summarize(rows):
    df = pd.DataFrame(rows)
    metric_cols = [
        "pre_entry_recall",
        "early_1s_recall",
        "event_precision",
        "event_f1",
        "false_alarms_per_min",
        "median_early_warning_s",
        "mean_early_warning_s",
    ]
    out = []
    for keys, group in df.groupby(["score_variant", "split", "threshold"], dropna=False):
        row = dict(zip(["score_variant", "split", "threshold"], keys))
        row["n_repeats"] = int(group["repeat_seed"].nunique())
        row["danger_videos_total"] = int(group["danger_videos"].sum())
        row["pre_entry_detected_total"] = int(group["pre_entry_detected"].sum())
        row["early_1s_detected_total"] = int(group["early_1s_detected"].sum())
        row["false_alarm_episodes_total"] = int(group["false_alarm_episodes"].sum())
        for col in metric_cols:
            values = pd.to_numeric(group[col], errors="coerce")
            row[f"{col}_mean"] = float(values.mean())
            row[f"{col}_std"] = float(values.std(ddof=0))
        out.append(row)
    return pd.DataFrame(out)


## Fonction `select_thresholds`

Cette cellule definit `select_thresholds`. Elle prepare une partie du script.

In [ ]:
def select_thresholds(summary, score_col):
    val = summary[(summary["split"] == "val") & (summary["score_variant"] == score_col)].copy()
    selected = {}
    for policy, order in POLICIES.items():
        table = val.copy()
        if policy == "low_false_alarm":
            eligible = table[table["false_alarms_per_min_mean"] <= 1.0]
            if eligible.empty:
                eligible = table
            table = eligible
            ascending = [True, False, False, False]
        elif policy == "max_1s_early_low_fa":
            ascending = [False, True, False, False]
        else:
            ascending = [False, False, True, False]
        sort_cols = [f"{col}_mean" for col in order]
        selected[policy] = table.sort_values(sort_cols, ascending=ascending).iloc[0].to_dict()
    return selected


## Fonction `run`

Cette cellule definit `run`. Elle prepare une partie du script.

In [ ]:
def run(args):
    final_score_run = resolve(args.final_score_run)
    run_dir = make_run_dir(args.run_name)
    df = load_final_scores(final_score_run)
    score_cols = args.score_cols or ["final_sequence_only"]
    thresholds = np.asarray(args.thresholds, dtype=float)
    seeds = sorted(df["repeat_seed"].unique().tolist())

    write_json(
        run_dir / "config.json",
        {
            "final_score_run": str(final_score_run),
            "score_cols": score_cols,
            "thresholds": [float(t) for t in thresholds],
            "persistence_windows": args.persistence_windows,
            "definition": "Causal early-warning audit: at time t, score is counted only if alarm starts before physical_entry_time_s. 1s-early recall requires first alarm <= entry_time - 1.0s.",
        },
    )

    rows = []
    for score_col in score_cols:
        for threshold in thresholds:
            for seed in seeds:
                for split_name in ["train", "val", "test"]:
                    row = evaluate_split(df, score_col, threshold, split_name, seed, args.persistence_windows)
                    if row is not None:
                        rows.append(row)

    detail = pd.DataFrame(rows)
    detail.to_csv(run_dir / "metrics" / "causal_early_warning_threshold_details.csv", index=False)
    summary = summarize(rows)
    summary.to_csv(run_dir / "metrics" / "causal_early_warning_threshold_summary.csv", index=False)

    selected_rows = []
    for score_col in score_cols:
        selected = select_thresholds(summary, score_col)
        for policy, val_row in selected.items():
            threshold = float(val_row["threshold"])
            test = summary[
                (summary["split"] == "test")
                & (summary["score_variant"] == score_col)
                & np.isclose(summary["threshold"].astype(float), threshold)
            ].iloc[0].to_dict()
            selected_rows.append(
                {
                    "score_variant": score_col,
                    "policy": policy,
                    "threshold": threshold,
                    "val_pre_entry_recall": val_row["pre_entry_recall_mean"],
                    "val_early_1s_recall": val_row["early_1s_recall_mean"],
                    "val_false_alarms_per_min": val_row["false_alarms_per_min_mean"],
                    "test_pre_entry_recall": test["pre_entry_recall_mean"],
                    "test_early_1s_recall": test["early_1s_recall_mean"],
                    "test_event_precision": test["event_precision_mean"],
                    "test_event_f1": test["event_f1_mean"],
                    "test_false_alarms_per_min": test["false_alarms_per_min_mean"],
                    "test_median_early_warning_s": test["median_early_warning_s_mean"],
                    "test_detected_total": test["pre_entry_detected_total"],
                    "test_early_1s_total": test["early_1s_detected_total"],
                    "test_danger_total": test["danger_videos_total"],
                    "test_false_alarm_episodes_total": test["false_alarm_episodes_total"],
                }
            )
    selected_df = pd.DataFrame(selected_rows)
    selected_df.to_csv(run_dir / "metrics" / "causal_early_warning_selected_policies.csv", index=False)

    lines = ["# Causal Early-Warning Audit", ""]
    lines.append("This is the live-stream style evaluation: each row is a decision at time `t`, using only pose history up to `t`. A detection only counts if the first alarm occurs before `physical_entry_time_s`; the 1s-early metric requires the alarm to occur at least one second before entry.")
    lines.append("")
    lines.append("| score | policy | threshold | pre-entry recall | >=1s early recall | precision | FA/min | median early s | detected | >=1s early | danger total |")
    lines.append("|---|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
    for _, row in selected_df.sort_values(["score_variant", "policy"]).iterrows():
        lines.append(
            f"| {row['score_variant']} | {row['policy']} | {row['threshold']:.2f} | "
            f"{row['test_pre_entry_recall']:.3f} | {row['test_early_1s_recall']:.3f} | "
            f"{row['test_event_precision']:.3f} | {row['test_false_alarms_per_min']:.3f} | "
            f"{row['test_median_early_warning_s']:.3f} | {int(row['test_detected_total'])} | "
            f"{int(row['test_early_1s_total'])} | {int(row['test_danger_total'])} |"
        )
    lines.append("")
    lines.append("Interpretation: these are the numbers to use for live early-warning claims. Subclip AP remains an offline chunk-ranking diagnostic, not proof of real-time prediction.")
    (run_dir / "causal_early_warning_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(run_dir)
    print(run_dir / "causal_early_warning_summary.md")


## Point d'entree principal

Cette cellule definit `main`. Elle prepare une partie du script.

In [ ]:
def main():
    parser = argparse.ArgumentParser(description="Causal pre-entry early-warning audit for final risk scores.")
    parser.add_argument("--final-score-run", default="runs/exp_088_physical_entry_final_score_full")
    parser.add_argument("--run-name", default="exp_092_causal_early_warning_audit")
    parser.add_argument("--score-cols", nargs="*", default=["final_sequence_only"])
    parser.add_argument("--thresholds", nargs="*", type=float, default=[round(x, 2) for x in np.arange(0.05, 1.00, 0.05)])
    parser.add_argument("--persistence-windows", type=int, default=2)
    args = parser.parse_args()
    run(args)


## Lancer le script

Cette cellule lance le `main()` avec des arguments adaptes au notebook.

In [ ]:
# Lancement du script
# Modifiez NOTEBOOK_ARGS si vous voulez changer les options.
from datetime import datetime
import sys

RUN_NAME_BASE = "exp_092_causal_early_warning_audit_notebook"
RUN_NAME = f"{RUN_NAME_BASE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
NOTEBOOK_ARGS = ["--run-name", RUN_NAME]

ancien_argv = sys.argv[:]
sys.argv = ["causal_early_warning_audit.py"] + NOTEBOOK_ARGS
try:
    print("Arguments utilises :", sys.argv[1:])
    main()
finally:
    sys.argv = ancien_argv
